<a href="https://colab.research.google.com/github/prasadpanigrahy/A-Python/blob/main/Exception.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Student Grade Management System
Program that records, updates, and deletes student grades.
Handle exceptions like invalid student ID, empty grade inputs, and type
mismatches.

In [ ]:
class StudentNotFoundException(Exception):
    pass

class InvalidGradeException(Exception):
    pass

class EmptyInputException(Exception):
    pass

class GradeManagementSystem:
    def __init__(self):
        self.students = {}  # {student_id: {'name': name, 'grades': {}}}

    def add_student(self, student_id, name):
        if not student_id or not name:
            raise EmptyInputException("Student ID and name cannot be empty")
        if student_id in self.students:
            raise ValueError("Student ID already exists")
        self.students[student_id] = {'name': name, 'grades': {}}
        print(f"Student {name} added successfully")

    def add_grade(self, student_id, subject, grade):
        if student_id not in self.students:
            raise StudentNotFoundException("Student ID not found")

        if not subject:
            raise EmptyInputException("Subject cannot be empty")

        try:
            grade = float(grade)
            if grade < 0 or grade > 100:
                raise InvalidGradeException("Grade must be between 0 and 100")
        except ValueError:
            raise InvalidGradeException("Grade must be a number")

        self.students[student_id]['grades'][subject] = grade
        print(f"Grade added for {self.students[student_id]['name']} in {subject}")

    def delete_student(self, student_id):
        if student_id not in self.students:
            raise StudentNotFoundException("Student ID not found")
        del self.students[student_id]
        print("Student deleted successfully")

    def display_students(self):
        if not self.students:
            print("No students in the system")
            return
        for sid, data in self.students.items():
            print(f"ID: {sid}, Name: {data['name']}, Grades: {data['grades']}")

# Test the system
def main():
    gms = GradeManagementSystem()

    while True:
        print("\n1. Add Student\n2. Add Grade\n3. Delete Student\n4. Display All\n5. Exit")
        choice = input("Enter choice: ")

        try:
            if choice == '1':
                sid = input("Enter student ID: ")
                name = input("Enter student name: ")
                gms.add_student(sid, name)

            elif choice == '2':
                sid = input("Enter student ID: ")
                subject = input("Enter subject: ")
                grade = input("Enter grade: ")
                gms.add_grade(sid, subject, grade)

            elif choice == '3':
                sid = input("Enter student ID to delete: ")
                gms.delete_student(sid)

            elif choice == '4':
                gms.display_students()

            elif choice == '5':
                break

            else:
                print("Invalid choice")

        except (StudentNotFoundException, InvalidGradeException,
                EmptyInputException, ValueError) as e:
            print(f"Error: {e}")

if __name__ == "__main__":
    main()

# Contact Book
Develop a contact book that can save, edit, and search contacts.
Handle errors such as duplicate entries, empty fields, and wrong phone number
format.

In [ ]:
import re

class DuplicateContactException(Exception):
    pass

class InvalidPhoneException(Exception):
    pass

class EmptyFieldException(Exception):
    pass

class ContactBook:
    def __init__(self):
        self.contacts = {}  # {name: {'phone': phone, 'email': email}}

    def validate_phone(self, phone):
        if not phone:
            raise EmptyFieldException("Phone number cannot be empty")

        # Accept formats: 1234567890, 123-456-7890, (123) 456-7890
        pattern = r'^\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}$'
        if not re.match(pattern, phone):
            raise InvalidPhoneException("Invalid phone number format")

        # Remove non-digits for storage
        return re.sub(r'\D', '', phone)

    def validate_email(self, email):
        if not email:
            return ""  # Email is optional
        pattern = r'^[\w\.-]+@[\w\.-]+\.\w+$'
        if not re.match(pattern, email):
            raise ValueError("Invalid email format")
        return email

    def add_contact(self, name, phone, email=""):
        if not name:
            raise EmptyFieldException("Name cannot be empty")

        if name in self.contacts:
            raise DuplicateContactException(f"Contact '{name}' already exists")

        valid_phone = self.validate_phone(phone)
        valid_email = self.validate_email(email)

        self.contacts[name] = {'phone': valid_phone, 'email': valid_email}
        print(f"Contact '{name}' added successfully")

    def edit_contact(self, name, phone=None, email=None):
        if name not in self.contacts:
            raise KeyError(f"Contact '{name}' not found")

        if phone:
            self.contacts[name]['phone'] = self.validate_phone(phone)
        if email is not None:  # Allow empty email
            self.contacts[name]['email'] = self.validate_email(email)

        print(f"Contact '{name}' updated successfully")

    def search_contact(self, keyword):
        if not keyword:
            raise EmptyFieldException("Search keyword cannot be empty")

        results = {name: details for name, details in self.contacts.items()
                  if keyword.lower() in name.lower()}

        if not results:
            print("No matching contacts found")
            return

        print("\nSearch Results:")
        for name, details in results.items():
            print(f"Name: {name}, Phone: {details['phone']}, Email: {details['email']}")

    def display_all(self):
        if not self.contacts:
            print("Contact book is empty")
            return

        print("\nAll Contacts:")
        for name, details in self.contacts.items():
            print(f"Name: {name}, Phone: {details['phone']}, Email: {details['email']}")

# Test the contact book
def main():
    book = ContactBook()

    while True:
        print("\n1. Add Contact\n2. Edit Contact\n3. Search Contact\n4. Display All\n5. Exit")
        choice = input("Enter choice: ")

        try:
            if choice == '1':
                name = input("Enter name: ")
                phone = input("Enter phone: ")
                email = input("Enter email (optional): ")
                book.add_contact(name, phone, email)

            elif choice == '2':
                name = input("Enter contact name to edit: ")
                print("Leave blank to keep current value")
                phone = input("New phone: ")
                email = input("New email: ")

                phone = phone if phone else None
                email = email if email else None
                book.edit_contact(name, phone, email)

            elif choice == '3':
                keyword = input("Enter name to search: ")
                book.search_contact(keyword)

            elif choice == '4':
                book.display_all()

            elif choice == '5':
                break
            else:
                print("Invalid choice")

        except (DuplicateContactException, InvalidPhoneException,
                EmptyFieldException, KeyError, ValueError) as e:
            print(f"Error: {e}")

if __name__ == "__main__":
    main()

# Banking System with Transactions
Simulate real-time transactions between bank accounts.
Handle errors like overdraft, transaction timeout, incorrect account numbers.

In [ ]:
import time
import random

class InsufficientFundsException(Exception):
    pass

class InvalidAccountException(Exception):
    pass

class TransactionTimeoutException(Exception):
    pass

class NegativeAmountException(Exception):
    pass

class Bank:
    def __init__(self):
        self.accounts = {
            '1001': {'name': 'Alice', 'balance': 5000},
            '1002': {'name': 'Bob', 'balance': 3000},
            '1003': {'name': 'Charlie', 'balance': 1000}
        }

    def validate_account(self, acc_no):
        if acc_no not in self.accounts:
            raise InvalidAccountException(f"Account {acc_no} not found")
        return True

    def transfer(self, from_acc, to_acc, amount):
        # Simulate network delay
        time.sleep(1)

        # Simulate random timeout (20% chance)
        if random.random() < 0.2:
            raise TransactionTimeoutException("Transaction timed out. Please try again")

        # Validate accounts
        self.validate_account(from_acc)
        self.validate_account(to_acc)

        # Validate amount
        if amount <= 0:
            raise NegativeAmountException("Amount must be positive")

        # Check sufficient funds
        if self.accounts[from_acc]['balance'] < amount:
            raise InsufficientFundsException(f"Insufficient funds. Available: ${self.accounts[from_acc]['balance']}")

        # Process transaction
        self.accounts[from_acc]['balance'] -= amount
        self.accounts[to_acc]['balance'] += amount

        print(f"Transaction successful!")
        print(f"${amount} transferred from {from_acc} to {to_acc}")
        print(f"New balance for {from_acc}: ${self.accounts[from_acc]['balance']}")

    def check_balance(self, acc_no):
        self.validate_account(acc_no)
        acc = self.accounts[acc_no]
        print(f"Account: {acc_no}, Name: {acc['name']}, Balance: ${acc['balance']}")

    def display_all(self):
        print("\nAll Accounts:")
        for acc_no, details in self.accounts.items():
            print(f"{acc_no}: {details['name']} - ${details['balance']}")

# Test the banking system
def main():
    bank = Bank()

    while True:
        print("\n1. Transfer Money\n2. Check Balance\n3. Display All\n4. Exit")
        choice = input("Enter choice: ")

        try:
            if choice == '1':
                from_acc = input("From account: ")
                to_acc = input("To account: ")
                amount = float(input("Amount: $"))
                bank.transfer(from_acc, to_acc, amount)

            elif choice == '2':
                acc_no = input("Enter account number: ")
                bank.check_balance(acc_no)

            elif choice == '3':
                bank.display_all()

            elif choice == '4':
                break
            else:
                print("Invalid choice")

        except (InsufficientFundsException, InvalidAccountException,
                TransactionTimeoutException, NegativeAmountException,
                ValueError) as e:
            print(f"Error: {e}")

if __name__ == "__main__":
    main()

# E-Commerce Order Management
Manage orders, returns, and refunds.
Handle cases like invalid coupon code, out-of-stock errors, invalid payment
methods.

In [ ]:
class OutOfStockException(Exception):
    pass

class InvalidCouponException(Exception):
    pass

class InvalidPaymentException(Exception):
    pass

class OrderException(Exception):
    pass

class ECommerce:
    def __init__(self):
        self.inventory = {
            'Laptop': {'price': 1000, 'stock': 5},
            'Phone': {'price': 500, 'stock': 10},
            'Headphones': {'price': 100, 'stock': 0}
        }
        self.valid_coupons = {'SAVE10': 10, 'SAVE20': 20}
        self.payment_methods = ['credit_card', 'paypal', 'cash']
        self.orders = {}
        self.order_id = 1

    def check_stock(self, product, quantity):
        if product not in self.inventory:
            raise OrderException(f"Product '{product}' not found")
        if self.inventory[product]['stock'] < quantity:
            raise OutOfStockException(f"{product} out of stock. Available: {self.inventory[product]['stock']}")
        return True

    def validate_coupon(self, coupon):
        if coupon and coupon not in self.valid_coupons:
            raise InvalidCouponException(f"Invalid coupon code: {coupon}")
        return self.valid_coupons.get(coupon, 0)

    def validate_payment(self, method):
        if method not in self.payment_methods:
            raise InvalidPaymentException(f"Invalid payment method. Use: {self.payment_methods}")
        return True

    def place_order(self, product, quantity, payment_method, coupon=None):
        # Check stock
        self.check_stock(product, quantity)

        # Validate payment
        self.validate_payment(payment_method)

        # Apply coupon
        discount = self.validate_coupon(coupon)

        # Calculate total
        total = self.inventory[product]['price'] * quantity
        discount_amount = total * discount / 100
        final_total = total - discount_amount

        # Update inventory
        self.inventory[product]['stock'] -= quantity

        # Create order
        order = {
            'id': self.order_id,
            'product': product,
            'quantity': quantity,
            'total': final_total,
            'payment': payment_method,
            'status': 'confirmed'
        }
        self.orders[self.order_id] = order
        self.order_id += 1

        print(f"Order placed successfully! Order ID: {order['id']}")
        print(f"Total: ${final_total} (Saved: ${discount_amount})")
        return order['id']

    def process_return(self, order_id):
        if order_id not in self.orders:
            raise OrderException(f"Order {order_id} not found")

        order = self.orders[order_id]
        if order['status'] != 'confirmed':
            raise OrderException("Order already processed")

        # Restore inventory
        self.inventory[order['product']]['stock'] += order['quantity']

        # Update order status
        order['status'] = 'returned'
        print(f"Order {order_id} returned successfully. Refund processed: ${order['total']}")

    def display_inventory(self):
        print("\nInventory:")
        for product, details in self.inventory.items():
            print(f"{product}: ${details['price']} - Stock: {details['stock']}")

# Test the system
def main():
    shop = ECommerce()

    while True:
        print("\n1. Place Order\n2. Return Order\n3. View Inventory\n4. Exit")
        choice = input("Enter choice: ")

        try:
            if choice == '1':
                product = input("Product: ")
                quantity = int(input("Quantity: "))
                payment = input("Payment method (credit_card/paypal/cash): ")
                coupon = input("Coupon (optional): ") or None

                shop.place_order(product, quantity, payment, coupon)

            elif choice == '2':
                order_id = int(input("Order ID to return: "))
                shop.process_return(order_id)

            elif choice == '3':
                shop.display_inventory()

            elif choice == '4':
                break
            else:
                print("Invalid choice")

        except (OutOfStockException, InvalidCouponException,
                InvalidPaymentException, OrderException, ValueError) as e:
            print(f"Error: {e}")

if __name__ == "__main__":
    main()

#Custom Exception Framework
Create your own custom exceptions for a specific application (like an Inventory
Management System).
For example: OutOfStockError, InvalidProductIDError, etc.

In [ ]:
class OutOfStockError(Exception):
    pass

class InvalidProductIDError(Exception):
    pass

class LowStockWarningError(Exception):
    pass

class NegativeQuantityError(Exception):
    pass

class ExpiredProductError(Exception):
    pass

class InventorySystem:
    def __init__(self):
        self.products = {
            'P001': {'name': 'Laptop', 'price': 1000, 'stock': 5, 'expiry': None},
            'P002': {'name': 'Milk', 'price': 3, 'stock': 2, 'expiry': '2024-01-15'},
            'P003': {'name': 'Bread', 'price': 2, 'stock': 0, 'expiry': '2024-01-10'}
        }

    def validate_product(self, product_id):
        if product_id not in self.products:
            raise InvalidProductIDError(f"Product ID '{product_id}' not found")
        return True

    def check_stock(self, product_id, quantity=1):
        self.validate_product(product_id)

        if quantity <= 0:
            raise NegativeQuantityError("Quantity must be positive")

        available = self.products[product_id]['stock']

        if available == 0:
            raise OutOfStockError(f"{self.products[product_id]['name']} is out of stock")

        if available < quantity:
            raise OutOfStockError(f"Insufficient stock. Available: {available}")

        if available < 5 and available > 0:
            print(f"Warning: {self.products[product_id]['name']} stock is low ({available} left)")

        return True

    def sell_product(self, product_id, quantity=1):
        self.check_stock(product_id, quantity)

        # Process sale
        self.products[product_id]['stock'] -= quantity
        product = self.products[product_id]

        print(f"Sold {quantity} {product['name']}(s)")
        print(f"Remaining stock: {product['stock']}")

        # Check if now out of stock
        if product['stock'] == 0:
            raise LowStockWarningError(f"{product['name']} is now out of stock!")

    def restock_product(self, product_id, quantity):
        self.validate_product(product_id)

        if quantity <= 0:
            raise NegativeQuantityError("Restock quantity must be positive")

        self.products[product_id]['stock'] += quantity
        print(f"Restocked {self.products[product_id]['name']}. New stock: {self.products[product_id]['stock']}")

    def check_expiry(self, product_id):
        self.validate_product(product_id)

        if self.products[product_id]['expiry']:
            import datetime
            today = datetime.date.today().isoformat()
            if self.products[product_id]['expiry'] < today:
                raise ExpiredProductError(f"{self.products[product_id]['name']} has expired!")
        return True

    def display_inventory(self):
        print("\nCurrent Inventory:")
        print("ID\tName\tPrice\tStock\tExpiry")
        for pid, details in self.products.items():
            print(f"{pid}\t{details['name']}\t${details['price']}\t{details['stock']}\t{details['expiry'] or 'N/A'}")

# Test the system
def main():
    inv = InventorySystem()

    while True:
        print("\n1. Sell Product\n2. Restock Product\n3. Check Expiry\n4. View Inventory\n5. Exit")
        choice = input("Enter choice: ")

        try:
            if choice == '1':
                pid = input("Product ID: ")
                qty = int(input("Quantity: "))
                inv.sell_product(pid, qty)

            elif choice == '2':
                pid = input("Product ID: ")
                qty = int(input("Quantity to add: "))
                inv.restock_product(pid, qty)

            elif choice == '3':
                pid = input("Product ID: ")
                inv.check_expiry(pid)
                print("Product is fresh")

            elif choice == '4':
                inv.display_inventory()

            elif choice == '5':
                break
            else:
                print("Invalid choice")

        except (OutOfStockError, InvalidProductIDError,
                LowStockWarningError, NegativeQuantityError,
                ExpiredProductError) as e:
            print(f"Error: {e}")

if __name__ == "__main__":
    main()

# Flight Booking System
A system to search, book, and cancel flight tickets.
Handle exceptions such as seat not available, invalid passenger details, payment
failure.

In [ ]:
class SeatNotAvailableError(Exception):
    pass

class InvalidPassengerError(Exception):
    pass

class PaymentFailedError(Exception):
    pass

class FlightNotFoundError(Exception):
    pass

class BookingSystem:
    def __init__(self):
        self.flights = {
            'AI101': {'from': 'New York', 'to': 'London', 'seats': 50, 'price': 500},
            'BA202': {'from': 'London', 'to': 'Paris', 'seats': 2, 'price': 200},
            'DL303': {'from': 'Chicago', 'to': 'Miami', 'seats': 0, 'price': 300}
        }
        self.bookings = {}
        self.booking_id = 1

    def validate_passenger(self, name, age, passport):
        if not name or len(name.strip()) < 2:
            raise InvalidPassengerError("Invalid name")
        if not age or not str(age).isdigit() or int(age) < 0 or int(age) > 120:
            raise InvalidPassengerError("Invalid age")
        if not passport or len(passport) < 5:
            raise InvalidPassengerError("Invalid passport number")
        return True

    def search_flights(self, from_city, to_city):
        results = []
        for fid, details in self.flights.items():
            if details['from'].lower() == from_city.lower() and details['to'].lower() == to_city.lower():
                results.append((fid, details))

        if not results:
            print("No flights found")
        else:
            print("\nAvailable Flights:")
            for fid, details in results:
                status = "Available" if details['seats'] > 0 else "Full"
                print(f"{fid}: {details['from']} → {details['to']} | ${details['price']} | Seats: {details['seats']} | {status}")
        return results

    def book_ticket(self, flight_id, passenger_name, age, passport, payment_method):
        # Check flight exists
        if flight_id not in self.flights:
            raise FlightNotFoundError(f"Flight {flight_id} not found")

        # Check seat availability
        if self.flights[flight_id]['seats'] <= 0:
            raise SeatNotAvailableError("No seats available on this flight")

        # Validate passenger
        self.validate_passenger(passenger_name, age, passport)

        # Validate payment method
        valid_payments = ['credit_card', 'debit_card', 'paypal']
        if payment_method not in valid_payments:
            raise PaymentFailedError(f"Invalid payment method. Use: {valid_payments}")

        # Simulate payment processing
        if payment_method == 'credit_card':
            # Simulate random payment failure (20% chance)
            import random
            if random.random() < 0.2:
                raise PaymentFailedError("Payment declined by bank")

        # Book ticket
        self.flights[flight_id]['seats'] -= 1

        booking = {
            'id': self.booking_id,
            'flight': flight_id,
            'passenger': passenger_name,
            'age': age,
            'passport': passport,
            'price': self.flights[flight_id]['price'],
            'status': 'confirmed'
        }
        self.bookings[self.booking_id] = booking
        self.booking_id += 1

        print(f"\n✅ Booking confirmed! Booking ID: {booking['id']}")
        print(f"Flight: {flight_id} | Passenger: {passenger_name}")
        print(f"Total paid: ${booking['price']}")
        return booking['id']

    def cancel_booking(self, booking_id):
        if booking_id not in self.bookings:
            raise FlightNotFoundError(f"Booking {booking_id} not found")

        booking = self.bookings[booking_id]
        if booking['status'] == 'cancelled':
            raise SeatNotAvailableError("Booking already cancelled")

        # Restore seat
        flight_id = booking['flight']
        self.flights[flight_id]['seats'] += 1

        # Update booking
        booking['status'] = 'cancelled'

        print(f"\n✅ Booking {booking_id} cancelled. Refund processed: ${booking['price']}")

# Test the system
def main():
    system = BookingSystem()

    while True:
        print("\n1. Search Flights")
        print("2. Book Ticket")
        print("3. Cancel Booking")
        print("4. Exit")
        choice = input("Enter choice: ")

        try:
            if choice == '1':
                from_city = input("From city: ")
                to_city = input("To city: ")
                system.search_flights(from_city, to_city)

            elif choice == '2':
                flight_id = input("Flight ID: ").upper()
                name = input("Passenger name: ")
                age = input("Age: ")
                passport = input("Passport: ")
                payment = input("Payment method (credit_card/debit_card/paypal): ")
                system.book_ticket(flight_id, name, age, passport, payment)

            elif choice == '3':
                booking_id = int(input("Booking ID: "))
                system.cancel_booking(booking_id)

            elif choice == '4':
                break
            else:
                print("Invalid choice")

        except (SeatNotAvailableError, InvalidPassengerError,
                PaymentFailedError, FlightNotFoundError, ValueError) as e:
            print(f"❌ Error: {e}")

if __name__ == "__main__":
    main()